# 11. Top three products per category, ties included

**Track:** sql | **Difficulty:** easy | **Tags:** top n per group, rank, dense_rank, row_number, window functions

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("011-top-products-with-ties")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "product_sales": """
UEFSMRUEFYAHFdoDTBVwFQASAADAAwhZGwAFAQBaDQgAWw0IAFwNCABdDQgAXg0IAF8NCABgDQgAYQ0IAGINCABjDQgAZA0IAGUN
CABmDQgAZw0IAGgNCABpDQgAag0IAGsNCABsDQgAbQ0IAG4NCABvDQgAcA0IAHENCAByDQgAcw0IAHQNCAB1DQgAdg0IAHcNCAB4
DQgAeQ0IAHoNCAB7DQgAfA0IAH0NCAB+DQgAfw0IAIANCACBDQgAgg0IAIMNCACEDQgAhQ0IAIYNCACHDQgAiA0IAIkNCACKDQgA
iw0IAIwNCACNDQgAjg0IPI8bAAAAAAAAkBsAAAAAAAAVABVkFWgsFXAVEBUGFQYcGAiQGwAAAAAAABgIWRsAAAAAAAAWACgIkBsA
AAAAAAAYCFkbAAAAAAAAEREAAAAyxAIAAABwAQYPQCAMRGEcSKIsTOM8UCRNVGVdWKZtXOd9YCiOZGmeaKqubOu+cCzPdG3fFQQV
ggIVhAJMFRYVABIAAIEB8FsKAAAATmV0d29ya2luZwcAAABDYW1lcmFzCwAAAEFjY2Vzc29yaWVzBgAAAFBob25lcwgAAABNb25p
dG9ycwcAAABTdG9yYWdlBQAAAEF1ZGlvBwAAAExhcHRvcAUfOFRhYmxldHMJAAAAV2VhcgEOMHMIAAAAUHJpbnRlcnMVABVIFUws
FXAVEBUGFQYcNgAoCVdlYXJhYmxlcxgLQWNjZXNzb3JpZXMREQAAACSMAgAAAHABBA8QMiNEJXZCQAUYIXBwAHlpmpU3hVORcyGE
qSWFFQQVphMV/gVMFXAVABIAANMJiBQAAABOZXR3b3JraW5nIHByb2R1Y3QgNhEAAABDYW1lcmFzFRU8MhUAAABBY2Nlc3Nvcmll
cxkZHBAAAABQaG9uHRQAM0oUAGJBACw0EgAAAE1vbml0b3IZV1YWAAA1AZsYU3RvcmFnZRmCXpsABDUPAbQMdWRpbxUsADEBQRRM
YXB0b3AZbAA1XkEAADFSmwAAM1o4AVazAAAyAXIAUzqzAAA3WkMAADMBLRRUYWJsZXQZnwW0PpABBSo+FQAANF7JAAA2WnAABZ0+
DwFe4AAANwGdPi0AADRaWgAANVoYACAxEwAAAFdlYXIB5RnkUokAADJWLAAANEbXAQAyIakUUHJpbnRlXURabAAlUD6SAQA2VmwA
JWc+9AAANkrfAlJVAAUVPs8BTh0DADQhWz6TAEXtPuMBADVWqABOkwAFQD68AAWTPlUAADFeOAIAMyFYQpwDBUQ+1wAANFaZAAA1
AUJCmgEl7z7wAAAzXoYARXk+LgAlMz6FAAAyFQAVZBVoLBVwFRAVBhUGHDYAKBNXZWFyYWJsZXMgcHJvZHVjdCA2GBVBY2Nlc3Nv
cmllcyBwcm9kdWN0IDEREQAAADLEAgAAAHABBg9AIAxEYRxIoixM4zxQJE1UZV1Ypm1c531gKI5kaZ5oqq5s675wLM90bd8VBBWg
BRX2AkwVVBUAEgAA0AIEQwAJAQBQCQcIAB8DCQkEYgIJCADuDQgEIgEJEAD0DQgAQA0IAB4NMAAMDRAAlgknBAAsDRAAQQ0QAC0N
SABIDRAAFg0gALYNEAAHDRAA4A0IAGYNMADHDRAAkA0QAIcNMABtDRgAWg0QAJoNEADYDQgAfA0IAHgNIASEAwkJANwNSAD6CQ8E
AIQNEADSDRAAkA04AAgNGABBDRAAgA0QAMgNKABWDRA8GgMAAAAAAACBAAAAAAAAABUAFXAVdCwVcBUQFQYVBhwYCIQDAAAAAAAA
GAhBAAAAAAAAABYIKAiEAwAAAAAAABgIQQAAAAAAAAAREQAAADjcCAAAADgBA+4cAQM8Bg9AIAxEYRxIoiyM0RjOswQBFEkTVVnS
hWXaxnXeN4BBKI4BWZonmi4AAAAVBBlcNQAYBnNjaGVtYRUIABUEJQIYCnByb2R1Y3RfaWQAFQwlAhgIY2F0ZWdvcnklAEwcAAAA
FQwlAhgMcHJvZHVjdF9uYW1lJQBMHAAAABUEJQIYCnVuaXRzX3NvbGQAFnAZHBlMJgAcFQQZNQAGEBkYCnByb2R1Y3RfaWQVAhZw
FoIJFuAFJoIEJggcGAiQGwAAAAAAABgIWRsAAAAAAAAWACgIkBsAAAAAAAAYCFkbAAAAAAAAEREAGSwVBBUAFQIAFQAVEBUCADwp
BhkmAHAAAAAmABwVDBk1AAYQGRgIY2F0ZWdvcnkVAhZwFsgDFs4DJowIJugFHDYAKAlXZWFyYWJsZXMYC0FjY2Vzc29yaWVzEREA
GSwVBBUAFQIAFQAVEBUCADwWhgcZBhkmAHAAAAAmABwVDBk1AAYQGRgMcHJvZHVjdF9uYW1lFQIWcBawFRaMCCbUDya2CRw2ACgT
V2VhcmFibGVzIHByb2R1Y3QgNhgVQWNjZXNzb3JpZXMgcHJvZHVjdCAxEREAGSwVBBUAFQIAFQAVEBUCADwW5g8ZBhkmAHAAAAAm
ABwVBBk1AAYQGRgKdW5pdHNfc29sZBUCFnAWrgcWiAUm2BQmwhEcGAiEAwAAAAAAABgIQQAAAAAAAAAWCCgIhAMAAAAAAAAYCEEA
AAAAAAAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmCGgAAAAWqCkWcCYIFsIWABkcGAxBUlJPVzpzY2hlbWEYjAMvLy8vL3lBQkFB
QVFBQUFBQUFBS0FBd0FCZ0FGQUFnQUNnQUFBQUFCQkFBTUFBQUFDQUFJQUFBQUJBQUlBQUFBQkFBQUFBUUFBQUMwQUFBQWNBQUFB
RHdBQUFBRUFBQUFiUC8vL3dBQUFRSVFBQUFBSEFBQUFBUUFBQUFBQUFBQUNnQUFBSFZ1YVhSelgzTnZiR1FBQUZ6Ly8vOEFBQUFC
UUFBQUFLRC8vLzhBQUFFRkVBQUFBQ0FBQUFBRUFBQUFBQUFBQUF3QUFBQndjbTlrZFdOMFgyNWhiV1VBQUFBQTFQLy8vOUQvLy84
QUFBRUZFQUFBQUNBQUFBQUVBQUFBQUFBQUFBZ0FBQUJqWVhSbFoyOXllUUFBQUFBRUFBUUFCQUFBQUJBQUZBQUlBQVlBQndBTUFB
QUFFQUFRQUFBQUFBQUJBaEFBQUFBa0FBQUFCQUFBQUFBQUFBQUtBQUFBY0hKdlpIVmpkRjlwWkFBQUNBQU1BQWdBQndBSUFBQUFB
QUFBQVVBQUFBQT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRlMHAAAHAAAHAAAHAAAAPgDAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "011-top-products-with-ties"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

product_sales = spark.read.parquet((_data_dir / "product_sales.parquet").resolve().as_posix())
product_sales.createOrReplaceTempView("product_sales")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'category': '1c4d82d3342ecabbdde2d454fb07856729cc0a8016e54353bab6eb67681a6a2c',
                         'product_id': 'f437b5864c97d5e47dd4fe81a859d4fbb55f1ab78fcd7c2166f7600697144419',
                         'product_name': '956ec03457a51d2945058b1d8e8f76fd0f2ff58a21588cb7c87dbd03742d27d1',
                         'sales_rank': '7580834e02133d80b710ad4e09e7c10569958190a27e91e1249eb14727365173',
                         'units_sold': '2ad22ed550d757e39fad6e570ee0c453098831a9f7d1823a7958feddb1bfded8'},
 'columns': [['category', 'string'],
             ['product_id', 'bigint'],
             ['product_name', 'string'],
             ['sales_rank', 'int'],
             ['units_sold', 'bigint']],
 'fingerprint': '9d9d5f8c9bf1e69b0d0e425b85782351097c73d8bda6aa769e3970e292bc1952',
 'order_matters': False,
 'row_count': 33}

check = make_check(EXPECTED)

## The situation

The merchandising team wants a "top three sellers" list for each product category.
Two products with the same number of units sold are equally good, so neither may be
dropped just because the list is "full".

## Input tables

### `product_sales`

One row per product.

| Column | Type | Description |
|---|---|---|
| `product_id` | BIGINT | Unique per product. |
| `category` | STRING | The product's category. |
| `product_name` | STRING | Display name. |
| `units_sold` | BIGINT | Units sold this quarter. NULL means no sales were recorded for the product. |

## Requirement

Return the top three products of every category by `units_sold`.

- The best-selling product of a category has `sales_rank` 1.
- Products with the same `units_sold` share the same rank. The next lower value gets
  the rank it would have had without the tie: after two products tied at rank 2, the
  next product is rank 4, not 3.
- Keep every product whose `sales_rank` is 3 or less. That can be more than three
  products when there is a tie, and fewer when the category is small.
- Products with a NULL `units_sold` are not ranked and do not appear.

## Expected output

One row per product that makes the list.

| Column | Type | Description |
|---|---|---|
| `category` | STRING | |
| `product_id` | BIGINT | |
| `product_name` | STRING | |
| `units_sold` | BIGINT | |
| `sales_rank` | INT | 1, 2 or 3. |

In [ ]:
product_sales.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `product_sales`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)